# Hot Zone Analysis

Investigate high-risk server/map/zone windows and identify the main pressure signals.

## Investigation checklist
- Find the highest-risk zones in the active analysis window.
- Check whether pressure is simulation, network, replication, physics, memory, or player impact.
- Use the output as evidence for Incident Dossier and Incident Workflow notes.

In [ ]:
import os
from pathlib import Path

import pandas as pd
import clickhouse_connect

CLICKHOUSE_HOST = os.getenv("CLICKHOUSE_HOST", "localhost")
CLICKHOUSE_PORT = int(os.getenv("CLICKHOUSE_PORT", "8123"))
CLICKHOUSE_DATABASE = os.getenv("CLICKHOUSE_DATABASE", "aegis_telemetry")
CLICKHOUSE_USERNAME = os.getenv("CLICKHOUSE_USERNAME", "default")
CLICKHOUSE_PASSWORD = os.getenv("CLICKHOUSE_PASSWORD", "aegis_dev_password")

client = clickhouse_connect.get_client(
    host=CLICKHOUSE_HOST,
    port=CLICKHOUSE_PORT,
    database=CLICKHOUSE_DATABASE,
    username=CLICKHOUSE_USERNAME,
    password=CLICKHOUSE_PASSWORD,
)

def run_query(sql: str) -> pd.DataFrame:
    return client.query_df(sql)

def load_sql(path: str) -> str:
    return Path(path).read_text(encoding="utf-8")

In [ ]:
# Edit these values for your investigation.
ANALYSIS_WINDOW_MINUTES = 60
ROW_LIMIT = 1000
SOURCE_PROFILE = "ALL"
REGION = "ALL"
SERVER_ID = "ALL"

def sql_quote(value: str) -> str:
    return "'" + str(value).replace("\\", "\\\\").replace("'", "\\'") + "'"

def source_filter() -> str:
    return "1 = 1" if SOURCE_PROFILE == "ALL" else f"source_profile = {sql_quote(SOURCE_PROFILE)}"

def region_filter() -> str:
    return "1 = 1" if REGION == "ALL" else f"region = {sql_quote(REGION)}"

def server_filter() -> str:
    return "1 = 1" if SERVER_ID == "ALL" else f"server_id = {sql_quote(SERVER_ID)}"

template_values = {
    "time_filter": f"window_start >= now() - INTERVAL {ANALYSIS_WINDOW_MINUTES} MINUTE",
    "incident_time_filter": f"detected_at >= now() - INTERVAL {ANALYSIS_WINDOW_MINUTES} MINUTE",
    "event_time_filter": f"event_time >= now() - INTERVAL {ANALYSIS_WINDOW_MINUTES} MINUTE",
    "quality_time_filter": f"failed_at >= now() - INTERVAL {ANALYSIS_WINDOW_MINUTES} MINUTE",
    "active_filter": f"{source_filter()} AND {region_filter()} AND {server_filter()}",
    "limit": str(int(ROW_LIMIT)),
}

In [ ]:
sql_template = load_sql("../sql/analyst_templates/hot_zone_summary.sql")
sql = sql_template.format(**template_values)
df = run_query(sql)
df.head(20)

In [ ]:
if not df.empty:
    numeric_cols = df.select_dtypes(include="number").columns.tolist()
    print(f"Rows: {len(df)}")
    print("Numeric columns:", numeric_cols)

    # Example: show the top rows by the first available numeric severity/risk column.
    preferred_sort_cols = [
        "max_hot_zone_risk",
        "p95_server_frame_ms",
        "incidents",
        "p95_packet_loss",
        "samples",
        "avg_confidence",
    ]
    sort_col = next((col for col in preferred_sort_cols if col in df.columns), numeric_cols[0] if numeric_cols else None)

    if sort_col:
        display(df.sort_values(sort_col, ascending=False).head(25))
else:
    print("No rows returned. Widen the analysis window or relax filters.")

## Next steps

- Export the dataframe with `df.to_csv(...)` if you need offline evidence.
- Compare source profiles, regions, maps, and zones before making a recommendation.
- Cross-check dashboard recommendations against raw evidence and guardrail metrics.
